# Reviews

Address comments.

In [ ]:
import ee
import geemap
from utils import *
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project = 'extents-490617')

config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder

In [ ]:
# there are objects defined in scripts 1 - 8 that will be used here. This requires running them in this script:

import nbimporter # lets you import notebooks like regular modules

%run 3_grids.ipynb # run full script here

In [ ]:
edge = ee.Image(f"{data_folder}/distance_to_secondary_edge").gt(30).rename("edge") # binary mask to identify edge pixels



# --- 1. Get projections ONCE outside the loop ---
sd_proj = biomass.projection()
age_proj = age.projection()

# Get SD pixel coordinates at SD resolution
sd_coords = ee.Image.pixelLonLat().reproject(sd_proj)

# Get age pixel coordinates at age resolution  
age_coords = ee.Image.pixelLonLat().reproject(age_proj)

# Reproject SD coords to age resolution (nearest neighbor = snap to parent center)
sd_lon_at_age = sd_coords.select('longitude').reproject(age_proj)
sd_lat_at_age = sd_coords.select('latitude').reproject(age_proj)

# Distance in degrees between age pixel center and its parent SD pixel center
d_lon = age_coords.select('longitude').subtract(sd_lon_at_age)
d_lat = age_coords.select('latitude').subtract(sd_lat_at_age)

# Convert to meters (approximate, valid near equator — fine for Amazon)
meters_per_deg_lon = ee.Image.constant(111320).multiply(
    age_coords.select('latitude').multiply(3.141592653589793 / 180).cos()
)
meters_per_deg_lat = ee.Image.constant(110540)

dx = d_lon.multiply(meters_per_deg_lon).abs()
dy = d_lat.multiply(meters_per_deg_lat).abs()

inner_mask = dy.lte(35).And(dx.lte(35))

over_1ha_features = ee.FeatureCollection("projects/forestregrowth/assets/secondary_polygons/secondary_age_vectors_022")



inner_patch = ee.Image(0)

proj = age.projection().getInfo()

# age = age.clip(amazon)
for year in range(1, 35):
    age_img = age.eq(year).unmask(0)

    sum_neighbors = age_img.focalMin(radius = 1, kernelType="square", units='pixels')

    inner_patch = inner_patch.add(sum_neighbors)

inner_patch = inner_patch.selfMask().reproject(age.projection())






# map = geemap.Map()
# map.addLayer(edge.updateMask(age), {'min':0, 'max':1, 'palette':['red', 'blue']}, 'edge')
# # map.addLayer(biomass, {'min':100, 'max':300, 'palette':['red', 'blue']}, 'biomass')
# # map.addLayer(inner_mask, {'min':0, 'max':1, 'palette':['black', 'white']}, 'inner_mask')
# # map.addLayer(over_1ha_features, {}, 'over_1ha_features')
# # map.addLayer(age, {'min':1, 'max':35, 'palette':['red', 'blue']}, 'age')
# # map.addLayer(inner_patch, {}, 'inner_patch')
# map

In [ ]:

# --- 1. Get projections ONCE outside the loop ---
sd_proj = biomass.projection()
age_proj = age.projection()

# Get SD pixel coordinates at SD resolution
sd_coords = ee.Image.pixelLonLat().reproject(sd_proj)

# Get age pixel coordinates at age resolution  
age_coords = ee.Image.pixelLonLat().reproject(age_proj)

# Reproject SD coords to age resolution (nearest neighbor = snap to parent center)
sd_lon_at_age = sd_coords.select('longitude').reproject(age_proj)
sd_lat_at_age = sd_coords.select('latitude').reproject(age_proj)

# Distance in degrees between age pixel center and its parent SD pixel center
d_lon = age_coords.select('longitude').subtract(sd_lon_at_age)
d_lat = age_coords.select('latitude').subtract(sd_lat_at_age)

# Convert to meters (approximate, valid near equator — fine for Amazon)
meters_per_deg_lon = ee.Image.constant(111320).multiply(
    age_coords.select('latitude').multiply(3.141592653589793 / 180).cos()
)
meters_per_deg_lat = ee.Image.constant(110540)

dx = d_lon.multiply(meters_per_deg_lon).abs()
dy = d_lat.multiply(meters_per_deg_lat).abs()

inner_mask = dy.lte(35).And(dx.lte(35))

In [ ]:
# keep only pixels surrounded by all pixels of the same age

inner_patch = ee.Image(0)

proj = age.projection().getInfo()

# age = age.clip(amazon)
for year in range(1, 35):
    age_img = age.eq(year).unmask(0)

    sum_neighbors = age_img.focalMin(radius = 1, kernelType="square", units='pixels')

    inner_patch = inner_patch.add(sum_neighbors)

inner_patch = inner_patch.selfMask().reproject(age.projection())

age_masked = age.updateMask(inner_mask).updateMask(inner_patch)
# create_grid(age_masked, region_name = "amazon", cell_size = 10000, file_name = "edge_removed_by_age_2")

NameError: name 'ee' is not defined

## Same-age patches

Keep only patches of the same age and greater than 1ha

Do not exclude edge pixels. Selecting exclusively based on area and age.

In [ ]:
grid = amazon.geometry().coveringGrid('EPSG:4326', 100000)

grid_list = grid.toList(grid.size())
n = grid.size().getInfo()

for i in range(n):
    cell = ee.Feature(grid_list.get(i))

    vectors = age.reduceToVectors(
        geometry=cell.geometry(),
        geometryType='polygon',
        scale=30,
        eightConnected=True,
        maxPixels=1e12,
        labelProperty='age',
        tileScale = 16
    )

    vectors = vectors.map(
        lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1),
            'tile_id': i + 1
        })
    ).filter(ee.Filter.gt('area_m2', 10000)).select(['age', 'tile_id'])

    task = ee.batch.Export.table.toAsset(
        collection=vectors,
        description=f"secondary_age_vectors_{i+1:03d}",
        assetId=f"{data_folder}/secondary_polygons/secondary_age_vectors_{i+1:03d}"
    )
    # task.start()

#309 needs to be run again

### Export age and biomass for ESA CCI for the same-age patches

In [ ]:
# for each collection, we will make a new collection by sampling one random pixel of age within it and returning it as a point feature.

def one_point_per_poly(f):
    geom = f.geometry()
    pt = ee.FeatureCollection.randomPoints(
        region = geom,
        points = 1,
        seed = ee.Number(1),   # or derive a deterministic seed if needed
        maxError = 1
    ).first()
    return ee.Feature(pt).copyProperties(f)

asset_list = ee.data.listAssets({'parent': f"{data_folder}/secondary_polygons"})['assets']

feature_ids = [a['name'] for a in asset_list]                    

for asset_id in feature_ids:
    polygons_fc = ee.FeatureCollection(asset_id)

    points_fc = ee.FeatureCollection(polygons_fc.map(one_point_per_poly))

    task = ee.batch.Export.table.toAsset(
        collection = points_fc,
        description = f"secondary_age_points_{asset_id[-3:]}",
        assetId = f"{data_folder}/secondary_points/secondary_age_vectors_{asset_id[-3:]}"
    )
    # task.start()


Try to get only patches with 9x9 same age

In [21]:
inner_patch = ee.Image(0)

proj = age.projection().getInfo()

age = age.clip(amazon)

for year in range(1, 35):
    age_img = age.eq(year).unmask(0)

    sum_neighbors = age_img.focalMin(radius = 30, kernelType="square", units='meters')

    inner_patch = inner_patch.add(sum_neighbors)

inner_patch = inner_patch.selfMask()

task = ee.batch.Export.image.toAsset(
    image=inner_patch,
    description="inner_patch",
    assetId=f"{data_folder}/inner_patch",
    region=roi,
    crs=proj['crs'],
    crsTransform=proj['transform'],
)
task.start()